In [10]:
import pandas as pd
from sqlalchemy import create_engine, text

# --- 1. Connect to Postgres ---
DB_USER = "postgres"
DB_PASSWORD = "1234"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "olist_retail"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

In [11]:
# --- 2. Load raw CSVs ---
raw_path = "../data/raw/"

customers = pd.read_csv(raw_path + "olist_customers_dataset.csv")
sellers = pd.read_csv(raw_path + "olist_sellers_dataset.csv")
category_translation = pd.read_csv(raw_path + "product_category_name_translation.csv")
products = pd.read_csv(raw_path + "olist_products_dataset.csv")
geolocation = pd.read_csv(raw_path + "olist_geolocation_dataset.csv")
orders = pd.read_csv(raw_path + "olist_orders_dataset.csv")
order_items = pd.read_csv(raw_path + "olist_order_items_dataset.csv")
order_payments = pd.read_csv(raw_path + "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(raw_path + "olist_order_reviews_dataset.csv")


In [12]:
# --- 3. Clean ---

# products: fill missing category with "unknown"
products["product_category_name"] = products["product_category_name"].fillna("unknown")

# category_translation: ensure every category in `products` has a matching row here,
# so the foreign key constraint doesn't break on load.
# This covers both the "unknown" placeholder and any categories missing from
# Olist's official translation table (discovered: pc_gamer, portateis_cozinha_e_preparadores_de_alimentos)
product_categories = set(products["product_category_name"].unique())
translation_categories = set(category_translation["product_category_name"].unique())
missing_categories = product_categories - translation_categories

if missing_categories:
    missing_df = pd.DataFrame({
        "product_category_name": list(missing_categories),
        "product_category_name_english": list(missing_categories)
    })
    category_translation = pd.concat([category_translation, missing_df], ignore_index=True)
    print(f"Added {len(missing_categories)} missing categories to translation table: {missing_categories}")

# geolocation: deduplicate down to one row per zip code prefix
geolocation = geolocation.drop_duplicates(subset="geolocation_zip_code_prefix", keep="first")

Added 3 missing categories to translation table: {'unknown', 'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}


In [13]:
# --- 4. Load order (respects foreign key dependencies) ---
truncate_order = [
    "order_reviews", "order_payments", "order_items",
    "orders", "products", "geolocation",
    "category_translation", "sellers", "customers"
]

with engine.begin() as conn:
    for table in truncate_order:
        conn.execute(text(f"TRUNCATE TABLE {table} CASCADE;"))
    print("Existing data truncated.")

load_order = [
    ("customers", customers),
    ("sellers", sellers),
    ("category_translation", category_translation),
    ("products", products),
    ("geolocation", geolocation),
    ("orders", orders),
    ("order_items", order_items),
    ("order_payments", order_payments),
    ("order_reviews", order_reviews),
]

for table_name, df in load_order:
    df.to_sql(table_name, engine, if_exists="append", index=False)
    print(f"Loaded {len(df)} rows into {table_name}")

print("ETL complete.")



Existing data truncated.
Loaded 99441 rows into customers
Loaded 3095 rows into sellers
Loaded 74 rows into category_translation
Loaded 32951 rows into products
Loaded 19015 rows into geolocation
Loaded 99441 rows into orders
Loaded 112650 rows into order_items
Loaded 103886 rows into order_payments
Loaded 99224 rows into order_reviews
ETL complete.
